# FCA-MoE — Colab T4 Research Prototype

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/msellamiTN/FCA-MoE/blob/main/notebooks/FCA_MoE_T4_Colab.ipynb)

This notebook validates the synthetic MoE/FCA vertical slice. It is not a pretrained LLM evaluation and does not demonstrate hard GPU isolation or physical multi-GPU speedup.

In [ ]:
from pathlib import Path
import os, sys, subprocess, platform
PROJECT_ROOT = Path('/content/FCA-MoE')
if not PROJECT_ROOT.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/msellamiTN/FCA-MoE.git',str(PROJECT_ROOT)],check=True)
os.chdir(PROJECT_ROOT); sys.path.insert(0,str(PROJECT_ROOT))
print('Python:',platform.python_version())

In [ ]:
import torch
print('PyTorch:',torch.__version__,'CUDA:',torch.version.cuda,'available:',torch.cuda.is_available())
if not torch.cuda.is_available(): raise RuntimeError('Select a GPU runtime in Colab.')
props=torch.cuda.get_device_properties(0); gpu_name=torch.cuda.get_device_name(0)
print('GPU:',gpu_name,'compute:',f'{props.major}.{props.minor}','VRAM GiB:',round(props.total_memory/2**30,2))
if 'T4' not in gpu_name.upper(): print('WARNING: assigned GPU is not T4; report its actual identity.')

In [ ]:
%cd /content/FCA-MoE
!pip -q install -e .
!python -m unittest discover -s tests -v

In [ ]:
from fca_moe.experiment import run_synthetic_experiment, save_json
result=run_synthetic_experiment(n_samples=4096,n_experts=8,d_model=32,epochs=30,seed=17,window_size=64,activation_rate_threshold=0.10)
save_json(result,'results/colab_t4_synthetic_run.json')
for key in ['device','gpu_name','torch_version','n_samples','n_experts','top_k','training_seconds','inference_seconds','fca_seconds','route_exact_match','concepts_returned','concept_enumeration_complete','virtual_placement_objective','virtual_placement_is_simulated']:
    print(key,':',result.get(key))

## Research interpretation gates
- Verify actual assigned device and software stack.
- Confirm exact FCA enumeration or explicitly report truncation.
- Compare FCA affinity with pairwise co-activation.
- Treat virtual placement as a simulation.
- No pretrained model is loaded in this notebook; GPT-OSS-20B remains a later candidate.
- Do not claim hard per-expert GPU isolation or multi-GPU all-to-all improvements from one T4.